# Customer Segmentation Analysis and Prediction

## 1. Introduction

This notebook outlines a comprehensive machine learning project focused on customer segmentation. Customer segmentation is the process of dividing customers into groups based on common characteristics. In the business world, understanding customer segments is crucial for targeted marketing, personalized product recommendations, and optimized customer service strategies.

The goal of this project is to build a classification model that can predict the `Segmentation` category (A, B, C, D) for new customers based on their demographic and behavioral attributes. We will cover data loading, exploratory data analysis (EDA), data preprocessing, feature engineering, model training, hyperparameter tuning, evaluation, and finally, model persistence.

**Dataset Schema:**
- `ID`: Unique customer identifier (int64)
- `Gender`: Gender of the customer (object)
- `Ever_Married`: Whether the customer is married (object)
- `Age`: Age of the customer (int64)
- `Graduated`: Whether the customer is a graduate (object)
- `Profession`: Profession of the customer (object)
- `Work_Experience`: Years of work experience (float64)
- `Spending_Score`: Spending behavior (object)
- `Family_Size`: Number of family members (float64)
- `Var_1`: Anonymized categorical variable (object)
- `Segmentation`: Customer segment (A, B, C, D) - **Target Variable** (object)

**Architecture Diagram:**

(Note: Please create the architecture diagram using https://app.diagrams.net/ and download it as an .svg file. A conceptual diagram involves: Data Source (CSV) -> Data Loading (Pandas) -> Preprocessing (Missing values, Outliers, Encoding, Scaling) -> Feature Engineering -> Data Splitting (Train/Test) -> Model Training (Algorithms, Hyperparameter Tuning, Imbalance Handling) -> Model Evaluation (Metrics, Visualizations) -> Model Persistence (Pickle) -> New Data Inference.)


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from collections import Counter
from imblearn.over_sampling import SMOTE
import pickle

import warnings
warnings.filterwarnings('ignore')

# Set display options for pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)


## 2. Data Loading

We will load the dataset from the specified CSV file ('Train.csv').


In [ ]:
# Load the dataset
try:
    df = pd.read_csv('Train.csv')
    print("Dataset loaded successfully.")
    print(f"Dataset shape: {df.shape}")
except FileNotFoundError:
    print("Error: 'Train.csv' not found. Please ensure the file is in the correct directory.")
    # Create a DataFrame from the sample data for demonstration if file not found
    sample_data = [
        {'ID': 462809, 'Gender': 'Male', 'Ever_Married': 'No', 'Age': 22, 'Graduated': 'No', 'Profession': 'Healthcare', 'Work_Experience': 1.0, 'Spending_Score': 'Low', 'Family_Size': 4.0, 'Var_1': 'Cat_4', 'Segmentation': 'D'},
        {'ID': 462643, 'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 38, 'Graduated': 'Yes', 'Profession': 'Engineer', 'Work_Experience': np.nan, 'Spending_Score': 'Average', 'Family_Size': 3.0, 'Var_1': 'Cat_4', 'Segmentation': 'A'},
        {'ID': 466315, 'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 67, 'Graduated': 'Yes', 'Profession': 'Engineer', 'Work_Experience': 1.0, 'Spending_Score': 'Low', 'Family_Size': 1.0, 'Var_1': 'Cat_6', 'Segmentation': 'B'},
        {'ID': 461735, 'Gender': 'Male', 'Ever_Married': 'Yes', 'Age': 67, 'Graduated': 'Yes', 'Profession': 'Lawyer', 'Work_Experience': 0.0, 'Spending_Score': 'High', 'Family_Size': 2.0, 'Var_1': 'Cat_6', 'Segmentation': 'B'},
        {'ID': 462669, 'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 40, 'Graduated': 'Yes', 'Profession': 'Entertainment', 'Work_Experience': np.nan, 'Spending_Score': 'High', 'Family_Size': 6.0, 'Var_1': 'Cat_6', 'Segmentation': 'A'},
        {'ID': 461319, 'Gender': 'Male', 'Ever_Married': 'Yes', 'Age': 56, 'Graduated': 'No', 'Profession': 'Artist', 'Work_Experience': 0.0, 'Spending_Score': 'Average', 'Family_Size': 2.0, 'Var_1': 'Cat_6', 'Segmentation': 'C'},
        {'ID': 460156, 'Gender': 'Male', 'Ever_Married': 'No', 'Age': 32, 'Graduated': 'Yes', 'Profession': 'Healthcare', 'Work_Experience': 1.0, 'Spending_Score': 'Low', 'Family_Size': 3.0, 'Var_1': 'Cat_6', 'Segmentation': 'C'},
        {'ID': 464347, 'Gender': 'Female', 'Ever_Married': 'No', 'Age': 33, 'Graduated': 'Yes', 'Profession': 'Healthcare', 'Work_Experience': 1.0, 'Spending_Score': 'Low', 'Family_Size': 3.0, 'Var_1': 'Cat_6', 'Segmentation': 'D'},
        {'ID': 465015, 'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 61, 'Graduated': 'Yes', 'Profession': 'Engineer', 'Work_Experience': 0.0, 'Spending_Score': 'Low', 'Family_Size': 3.0, 'Var_1': 'Cat_7', 'Segmentation': 'D'},
        {'ID': 465176, 'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 55, 'Graduated': 'Yes', 'Profession': 'Artist', 'Work_Experience': 1.0, 'Spending_Score': 'Average', 'Family_Size': 4.0, 'Var_1': 'Cat_6', 'Segmentation': 'C'}
    ]
    df = pd.DataFrame(sample_data)
    print("Loaded sample data for demonstration.")
    print(f"Sample data shape: {df.shape}")

# Display the first few rows of the DataFrame
print("\nFirst 5 rows of the dataset:")
print(df.head())

# Display basic information about the dataset
print("\nDataset Info:")
df.info()

# Display descriptive statistics
print("\nDescriptive Statistics for Numerical Features:")
print(df.describe())


## 3. Exploratory Data Analysis (EDA)

EDA helps us understand the dataset's characteristics, identify patterns, and detect potential issues like missing values or outliers.

### Missing Values Analysis

Identifying and handling missing values is a crucial step to ensure data quality.


In [ ]:
# Check for missing values
print("Missing values before imputation:")
missing_values = df.isnull().sum()
print(missing_values[missing_values > 0])

# Impute missing values
# For numerical features, we'll use the median due to potential outliers.
# For categorical features, we'll use the mode (most frequent category).

# Numerical features to impute
numerical_cols_with_nan = ['Work_Experience', 'Family_Size']
for col in numerical_cols_with_nan:
    if col in df.columns and df[col].isnull().any():
        median_val = df[col].median()
        df[col].fillna(median_val, inplace=True)
        print(f"Filled missing values in '{col}' with median: {median_val}")

# Categorical features to impute
# Identify categorical columns that might have NaNs based on previous output or schema
categorical_cols_with_nan = ['Profession', 'Graduated', 'Ever_Married', 'Var_1'] # Add other object columns if they show NaNs in `missing_values`
for col in categorical_cols_with_nan:
    if col in df.columns and df[col].isnull().any():
        mode_val = df[col].mode()[0]
        df[col].fillna(mode_val, inplace=True)
        print(f"Filled missing values in '{col}' with mode: {mode_val}")

# Verify no more missing values
print("\nMissing values after imputation:")
missing_after_imputation = df.isnull().sum()
print(missing_after_imputation[missing_after_imputation > 0])
if missing_after_imputation.sum() == 0:
    print("All missing values handled successfully.")


### Outlier Handling

Outliers can significantly affect model performance. We will identify and handle outliers primarily in numerical features using the IQR method.


In [ ]:
# Function to detect and handle outliers using IQR
def handle_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    if not outliers.empty:
        print(f"\nOutliers detected in '{column}': {len(outliers)} rows")
        # Cap the outliers (replace with bounds)
        df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
        df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
        print(f"Outliers in '{column}' have been capped at [{lower_bound:.2f}, {upper_bound:.2f}].")
    else:
        print(f"\nNo significant outliers detected in '{column}'.")
    return df

# Apply outlier handling to numerical features
numerical_features = ['Age', 'Work_Experience', 'Family_Size']
for col in numerical_features:
    df = handle_outliers_iqr(df, col)

# Re-check descriptive statistics after outlier handling
print("\nDescriptive statistics after outlier handling:")
print(df[numerical_features].describe())


## 4. Preprocessing

This step involves transforming raw data into a format suitable for machine learning models. This includes feature engineering, encoding categorical variables, and scaling numerical features.

### Feature Engineering

We can create new features that might provide more predictive power. For instance, `Age` can be grouped into bins.


In [ ]:
# Feature Engineering: Age Groups
df['Age_Group'] = pd.cut(df['Age'], bins=[0, 18, 30, 45, 60, 100],
                         labels=['0-18', '19-30', '31-45', '46-60', '60+'], right=True)
print("\n'Age_Group' feature created.")
print(df['Age_Group'].value_counts())

# Feature Engineering: Work_Experience_Per_Family_Member
# Adding a small constant to Family_Size to avoid division by zero
df['Work_Experience_Per_Family_Member'] = df['Work_Experience'] / (df['Family_Size'] + 1e-6)
print("\n'Work_Experience_Per_Family_Member' feature created.")
print(df[['Work_Experience', 'Family_Size', 'Work_Experience_Per_Family_Member']].head())


## 5. Visual Representation of EDA (Plotly)

Visualizations are key to understanding distributions, relationships, and characteristics of the data. We will use Plotly for interactive plots.


In [ ]:
# Set a custom color palette for consistent visualization
custom_colors = px.colors.qualitative.Plotly

# 1. Distribution of Numerical Features
print("Plotting distributions of numerical features...")
for col in ['Age', 'Work_Experience', 'Family_Size', 'Work_Experience_Per_Family_Member']:
    fig = px.histogram(df, x=col, marginal="box", title=f'Distribution of {col}',
                       color_discrete_sequence=custom_colors)
    fig.show()

# 2. Distribution of Categorical Features
print("\nPlotting distributions of categorical features...")
categorical_cols = ['Gender', 'Ever_Married', 'Graduated', 'Profession', 'Spending_Score', 'Family_Size', 'Var_1', 'Age_Group']
for col in categorical_cols:
    fig = px.bar(df[col].value_counts().reset_index(), x='index', y=col,
                 title=f'Count of {col}', labels={'index': col, col: 'Count'},
                 color_discrete_sequence=custom_colors)
    fig.update_xaxes(categoryorder='total ascending')
    fig.show()

# 3. Relationship between Features and Target (Segmentation)
print("\nPlotting relationships between features and 'Segmentation'...")
# Gender vs. Segmentation
fig = px.bar(df.groupby(['Gender', 'Segmentation']).size().reset_index(name='Count'),
             x='Gender', y='Count', color='Segmentation', barmode='group',
             title='Gender Distribution by Segmentation', color_discrete_sequence=custom_colors)
fig.show()

# Age Distribution by Segmentation
fig = px.box(df, x='Segmentation', y='Age', color='Segmentation',
             title='Age Distribution by Segmentation', color_discrete_sequence=custom_colors)
fig.show()

# Profession vs. Segmentation
fig = px.bar(df.groupby(['Profession', 'Segmentation']).size().reset_index(name='Count'),
             x='Profession', y='Count', color='Segmentation', barmode='group',
             title='Profession Distribution by Segmentation', color_discrete_sequence=custom_colors)
fig.update_xaxes(categoryorder='total descending')
fig.show()

# Spending Score vs. Segmentation (Ordinal)
spending_order = ['Low', 'Average', 'High']
df['Spending_Score'] = pd.Categorical(df['Spending_Score'], categories=spending_order, ordered=True)
fig = px.bar(df.groupby(['Spending_Score', 'Segmentation']).size().reset_index(name='Count'),
             x='Spending_Score', y='Count', color='Segmentation', barmode='group',
             title='Spending Score Distribution by Segmentation', category_orders={'Spending_Score': spending_order},
             color_discrete_sequence=custom_colors)
fig.show()

# Family Size vs. Segmentation
fig = px.box(df, x='Segmentation', y='Family_Size', color='Segmentation',
             title='Family Size Distribution by Segmentation', color_discrete_sequence=custom_colors)
fig.show()

# Insights from EDA Visualizations:
# - Age distribution appears somewhat normal, with a slight skew. Different segments might show distinct age patterns.
# - Work Experience and Family Size also show varied distributions, with some segments potentially having more experienced or larger families.
# - Gender distribution for segments seems relatively balanced, suggesting it might not be a primary driver alone.
# - 'Profession' and 'Spending_Score' show clear differences across segments. For example, 'Artist' and 'Healthcare' might be dominant in certain segments, and 'High' spending score customers might cluster in specific segments.
# - 'Var_1' also has distinct categories, and its distribution across segments could reveal hidden patterns.


## 6. Visual Representation of Correlation and Covariance

Correlation measures the strength and direction of a linear relationship between two numerical variables. Covariance also indicates the direction of the linear relationship but is not scaled, making correlation a more interpretable metric.


In [ ]:
# Select only numerical features for correlation matrix
numerical_df = df[numerical_features + ['Work_Experience_Per_Family_Member']]

# Calculate the correlation matrix
correlation_matrix = numerical_df.corr()
print("Correlation Matrix:")
print(correlation_matrix)

# Visualize the correlation matrix using a heatmap
fig = px.imshow(correlation_matrix, text_auto=True, aspect="auto",
                color_continuous_scale='RdBu_r',
                title='Correlation Matrix of Numerical Features')
fig.show()

# Explain Covariance:
# Covariance is a measure of how two variables change together.
# A positive covariance means that the variables move in the same direction (as one increases, the other tends to increase).
# A negative covariance means that they move in opposite directions.
# A covariance of zero means no linear relationship.
# Unlike correlation, covariance is not normalized, so its magnitude depends on the scales of the variables, making it harder to compare relationships across different pairs of variables directly.
# For example, let's look at the covariance between Age and Work_Experience:
covariance_age_work = df['Age'].cov(df['Work_Experience'])
print(f"\nCovariance between Age and Work_Experience: {covariance_age_work:.2f}")

# Explaining the plots:
# The heatmap above visualizes the correlation matrix.
# - Values close to 1 (bright red) indicate a strong positive linear correlation (e.g., as 'Age' increases, 'Work_Experience' tends to increase).
# - Values close to -1 (bright blue) indicate a strong negative linear correlation (e.g., as 'Age' increases, 'Work_Experience' tends to decrease).
# - Values close to 0 (white/light grey) indicate a weak or no linear correlation.
#
# From the plot, we can infer:
# - There is likely a positive correlation between `Age` and `Work_Experience`, which is intuitive.
# - `Work_Experience_Per_Family_Member` might have interesting correlations with other features, and its relation to `Family_Size` (negative) and `Work_Experience` (positive) is expected.
# - High correlations between independent variables (multicollinearity) can be an issue for some models (e.g., Logistic Regression), but ensemble methods like Random Forest are more robust to it.


## 7. Feature Selection Based on EDA

Based on our EDA, we will select features that are relevant for predicting customer segmentation.
- `ID`: This is just an identifier and should be dropped.
- `Gender`, `Ever_Married`, `Age`, `Graduated`, `Profession`, `Work_Experience`, `Spending_Score`, `Family_Size`, `Var_1`: All appear to be relevant and show variations across segments.
- `Age_Group`, `Work_Experience_Per_Family_Member`: Our engineered features also seem promising.


In [ ]:
# Drop the 'ID' column as it's not a predictive feature
df_processed = df.drop('ID', axis=1)
print(f"Dropped 'ID' column. New shape: {df_processed.shape}")

# Define features (X) and target (y)
X = df_processed.drop('Segmentation', axis=1)
y = df_processed['Segmentation']

print("\nSelected features for modeling (excluding 'ID' and target 'Segmentation'):")
print(X.columns.tolist())


## 8. Separate the Selected Features for Training

We need to split our dataset into training and testing sets. This ensures that we can evaluate the model's performance on unseen data, providing an unbiased assessment of its generalization capability.

The selected features (`Gender`, `Ever_Married`, `Age`, `Graduated`, `Profession`, `Work_Experience`, `Spending_Score`, `Family_Size`, `Var_1`, `Age_Group`, `Work_Experience_Per_Family_Member`) are taken because they represent various aspects of customer demographics and behavior that are likely to influence their segmentation. These features have shown distinct distributions and relationships with the target variable during EDA, making them valuable predictors.


In [ ]:
# Encode target variable 'Segmentation'
# LabelEncoder is suitable for the target variable in classification tasks.
le = LabelEncoder()
y_encoded = le.fit_transform(y)
print(f"Original Segmentation labels: {le.classes_}")
print(f"Encoded Segmentation labels: {np.unique(y_encoded)}")
# Store the mapping for later use
segmentation_mapping = dict(zip(le.classes_, le.transform(le.classes_)))
print(f"Segmentation mapping: {segmentation_mapping}")

# Split the data into training and testing sets
# We use stratify=y_encoded to ensure that the proportion of target classes is similar in both train and test sets.
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded)

print(f"\nShape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print("\nDistribution of target variable in training set:")
print(pd.Series(y_train).value_counts(normalize=True))
print("\nDistribution of target variable in test set:")
print(pd.Series(y_test).value_counts(normalize=True))

# Identify categorical and numerical features for preprocessing pipeline
numerical_features = ['Age', 'Work_Experience', 'Family_Size', 'Work_Experience_Per_Family_Member']
categorical_features_onehot = ['Gender', 'Profession', 'Var_1', 'Age_Group'] # High cardinality/nominal
categorical_features_ordinal = ['Ever_Married', 'Graduated', 'Spending_Score'] # Binary or ordinal

# Define the order for ordinal features
ever_married_order = ['No', 'Yes']
graduated_order = ['No', 'Yes']
spending_score_order = ['Low', 'Average', 'High']

# Create preprocessing pipelines for numerical and categorical features
# Numerical pipeline: Impute with median (already done, but keeping for robustness if data changes), then StandardScale
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# One-hot encode categorical features (for nominal variables)
onehot_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Ordinal encode categorical features (for binary/ordinal variables)
ordinal_transformer_ever_married = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[ever_married_order]))
])
ordinal_transformer_graduated = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[graduated_order]))
])
ordinal_transformer_spending_score = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[spending_score_order]))
])

# Create a preprocessor using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('onehot', onehot_transformer, categorical_features_onehot),
        ('ever_married', ordinal_transformer_ever_married, ['Ever_Married']),
        ('graduated', ordinal_transformer_graduated, ['Graduated']),
        ('spending_score', ordinal_transformer_spending_score, ['Spending_Score'])
    ],
    remainder='passthrough' # Keep any other columns not specified (e.g., 'ID' if not dropped)
)


## 9. Modeling

This is a multi-class classification problem as we are predicting customer `Segmentation` (A, B, C, D). We will use several appropriate classification models to find the best performer.

Models selected:
1.  **Logistic Regression**: A good baseline, interpretable, and computationally efficient. It models the probability of a binary outcome, extended for multi-class using One-vs-Rest or Multinomial approach.
2.  **Random Forest Classifier**: An ensemble method that builds multiple decision trees and merges their predictions. It's robust to overfitting, handles non-linear relationships, and implicitly performs feature selection.
3.  **XGBoost Classifier**: A highly optimized gradient boosting framework known for its speed and performance. It builds trees sequentially, with each new tree trying to correct errors of the previous ones.

We will also check for class imbalance in the target variable and address it if necessary using SMOTE.


In [ ]:
# Check for class imbalance in the training data
print("Class distribution in training set before SMOTE:")
print(Counter(y_train))

# Apply SMOTE if there's significant imbalance
# A rule of thumb for "significant imbalance" could be if the smallest class is < 10-20% of the largest class.
# Given the small sample size, we'll apply it cautiously, but it's crucial for larger datasets.
# For demonstration purposes, we will apply SMOTE if any class count is below 50% of the mean class count.
mean_class_count = np.mean(list(Counter(y_train).values()))
min_class_count = min(Counter(y_train).values())

if min_class_count < 0.5 * mean_class_count and len(np.unique(y_train)) > 1:
    print("\nApplying SMOTE for class imbalance...")
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(preprocessor.fit_transform(X_train), y_train)
    print("Class distribution in training set after SMOTE:")
    print(Counter(y_train_resampled))
else:
    print("\nClass distribution is relatively balanced or not enough data for SMOTE. Skipping SMOTE.")
    X_train_resampled = preprocessor.fit_transform(X_train)
    y_train_resampled = y_train

# Store the preprocessor fitted on training data for later use with test data and new data
fitted_preprocessor = preprocessor.fit(X_train) # Fit preprocessor again to ensure it captures all categories from train

# Transform X_train and X_test using the fitted preprocessor
X_train_transformed = fitted_preprocessor.transform(X_train)
X_test_transformed = fitted_preprocessor.transform(X_test)

# Define models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, multi_class='multinomial', solver='lbfgs', max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='mlogloss')
}

results = {}

print("\n--- Model Training and Initial Evaluation ---")
for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train_resampled if 'SMOTE' in globals() else X_train_transformed, y_train_resampled if 'SMOTE' in globals() else y_train)
    y_pred = model.predict(X_test_transformed)
    accuracy = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted') # Using weighted for multi-class
    results[name] = {'accuracy': accuracy, 'f1_score': f1, 'model': model}
    print(f"{name} - Accuracy: {accuracy:.4f}, F1-Score (weighted): {f1:.4f}")
    print(f"Classification Report for {name}:\n{classification_report(y_test, y_pred, target_names=le.classes_)}")


## 10. Evaluation Metrics

For multi-class classification, the following metrics are essential:
-   **Accuracy**: The proportion of correctly predicted instances out of the total instances. Useful if classes are balanced.
-   **Precision**: The ability of the classifier not to label as positive a sample that is negative. For each class, it's `TP / (TP + FP)`.
-   **Recall**: The ability of the classifier to find all the positive samples. For each class, it's `TP / (TP + FN)`.
-   **F1-Score**: The harmonic mean of precision and recall. It's a good measure when you need to balance both precision and recall, especially in imbalanced datasets.
-   **Confusion Matrix**: A table that describes the performance of a classification model on a set of test data for which the true values are known.

These metrics are suitable because they provide a comprehensive view of the model's performance beyond just overall accuracy, especially detailing how well the model performs for each individual class. `f1_score(average='weighted')` is used to account for class imbalance when averaging across classes.


## 11. Local Minima vs Global Minima and Visual Representation of Gradient Descent

### Local Minima vs. Global Minima

In the context of machine learning, especially when training models using optimization algorithms like gradient descent, we are trying to find the set of parameters (weights and biases) that minimize a cost function (or loss function).

-   **Global Minimum**: This is the lowest possible value of the cost function across its entire domain. It represents the absolute best set of parameters for the model to make predictions.
-   **Local Minimum**: This is a point where the cost function is lower than all its neighboring points within a certain region, but not necessarily the lowest value across the entire domain. An optimization algorithm might get stuck in a local minimum if it doesn't have mechanisms to escape it.

For convex cost functions (like in Logistic Regression), there's only one global minimum, making optimization straightforward. However, for complex models (e.g., neural networks), the cost surface can be non-convex with many local minima, posing challenges for finding the truly optimal parameters.

### Visual Representation of Gradient Descent

Gradient Descent is an iterative optimization algorithm used to find the minimum of a function. It works by taking repeated steps in the opposite direction of the gradient (or steepest ascent) of the function at the current point.

Let's imagine a simple 2D cost function `J(θ)` where `θ` is a single parameter.


In [ ]:
# Conceptual visualization of Gradient Descent
# This is a simplified 2D example for illustrative purposes, not directly using our high-dimensional dataset.

def cost_function(theta):
    # A simple quadratic function for illustration (convex, has one global minimum)
    return (theta - 5)**2 + 10

def gradient(theta):
    # Derivative of the cost function
    return 2 * (theta - 5)

# Initial parameter
theta_start = 0
learning_rate = 0.1
iterations = 20

theta_history = [theta_start]
cost_history = [cost_function(theta_start)]

for i in range(iterations):
    grad_val = gradient(theta_history[-1])
    new_theta = theta_history[-1] - learning_rate * grad_val
    theta_history.append(new_theta)
    cost_history.append(cost_function(new_theta))

# Generate points for the cost function curve
theta_values = np.linspace(0, 10, 100)
cost_values = cost_function(theta_values)

fig = go.Figure()

# Plot the cost function curve
fig.add_trace(go.Scatter(x=theta_values, y=cost_values, mode='lines', name='Cost Function J(theta)',
                         line=dict(color='blue')))

# Plot the gradient descent path
fig.add_trace(go.Scatter(x=theta_history, y=cost_history, mode='lines+markers', name='Gradient Descent Path',
                         marker=dict(size=8, color='red'), line=dict(color='red', dash='dot')))

# Mark the global minimum
fig.add_trace(go.Scatter(x=[5], y=[10], mode='markers', name='Global Minimum',
                         marker=dict(size=10, color='green', symbol='star')))

fig.update_layout(title='Conceptual Visualization of Gradient Descent',
                  xaxis_title='Parameter (theta)',
                  yaxis_title='Cost J(theta)',
                  showlegend=True)
fig.show()

# Explanation for the plot:
# The blue curve represents a simple cost function J(theta). Our goal is to find the 'theta' that minimizes J(theta).
# The red dashed line with markers shows the path taken by the gradient descent algorithm.
# - Starting from an initial `theta_start`, the algorithm iteratively updates `theta` by moving in the direction opposite to the gradient.
# - Each step reduces the cost, bringing `theta` closer to the minimum.
# - The `learning_rate` controls the size of each step. A too large learning rate can overshoot the minimum, while a too small one can make convergence very slow.
# The green star marks the global minimum, which gradient descent successfully approaches in this convex example.
# For our classification models like Logistic Regression, the underlying optimization uses variants of gradient descent (e.g., L-BFGS, Stochastic Gradient Descent) to find the optimal weights for features.


## 12. Residuals and How to Visualize Them

### Residuals

In classification, residuals are not as directly interpretable as in regression (where they are the difference between actual and predicted *continuous* values). For classification, a common way to think about "residuals" is in terms of misclassifications or the confidence of predictions for incorrect classes.

-   **For Classification**: A residual could be conceptualized as the difference between the true class and the predicted class, or more informatively, the difference between the true class and the predicted *probability* of that class.
    -   **Misclassified Points**: Simply identifying the data points that were predicted incorrectly.
    -   **Probability Distribution for Each Class**: Visualizing the predicted probabilities for each class against the true labels can show how "confident" the model was when it made a correct or incorrect prediction.

### How to Visualize Residuals in Classification

We can visualize misclassifications directly using a confusion matrix or by plotting predicted probabilities.


In [ ]:
# Re-evaluate the best model (e.g., Random Forest)
best_model_name = max(results, key=lambda k: results[k]['f1_score'])
best_model = results[best_model_name]['model']

y_pred_best = best_model.predict(X_test_transformed)
y_pred_proba_best = best_model.predict_proba(X_test_transformed)

# 1. Visualization via Confusion Matrix (most common for classification "residuals")
cm = confusion_matrix(y_test, y_pred_best)
fig_cm = px.imshow(cm, text_auto=True, color_continuous_scale='Blues',
                   labels=dict(x="Predicted", y="True"),
                   x=le.classes_, y=le.classes_,
                   title=f'Confusion Matrix for {best_model_name}')
fig_cm.show()

# Explanation of Confusion Matrix:
# - Each row represents the instances in an actual class.
# - Each column represents the instances in a predicted class.
# - The diagonal elements show the number of correctly classified instances for each class (True Positives).
# - Off-diagonal elements show misclassifications (False Positives and False Negatives).
# - For example, if Segment 'A' is predicted as 'B', it means the model incorrectly assigned an 'A' customer to segment 'B'. This helps us understand specific types of errors.

# 2. Visualizing predicted probabilities (conceptual "residuals" for misclassified points)
# Let's take a sample of misclassified points and visualize their predicted probabilities
misclassified_indices = np.where(y_test != y_pred_best)[0]
if len(misclassified_indices) > 0:
    print(f"\nNumber of misclassified points: {len(misclassified_indices)}")
    sample_misclassified_indices = misclassified_indices[:min(10, len(misclassified_indices))] # Take up to 10 samples

    print("\nPredicted probabilities for some misclassified samples:")
    for idx_in_test in sample_misclassified_indices:
        true_label = le.inverse_transform([y_test[idx_in_test]])[0]
        predicted_label = le.inverse_transform([y_pred_best[idx_in_test]])[0]
        predicted_probs = y_pred_proba_best[idx_in_test]

        print(f"True: {true_label}, Predicted: {predicted_label}")
        for i, class_name in enumerate(le.classes_):
            print(f"  P({class_name}): {predicted_probs[i]:.4f}")

        # Plotting probabilities for a single misclassified sample
        fig_prob = px.bar(x=le.classes_, y=predicted_probs,
                          title=f'Predicted Probabilities for a Misclassified Sample (True: {true_label}, Pred: {predicted_label})',
                          labels={'x': 'Class', 'y': 'Probability'},
                          color_discrete_sequence=custom_colors)
        fig_prob.show()
else:
    print("No misclassified points to visualize probabilities for.")


# Comparing metrics to suggest improvements:
# - If **Accuracy is low but F1-score is high for certain classes**: The model is good at predicting those specific classes, but overall performance is dragged down by other classes or class imbalance.
# - If **Precision is low for a class**: The model is predicting too many false positives for that class. It's wrongly identifying other segments as this one. We might need to refine features that distinguish this class from others.
# - If **Recall is low for a class**: The model is missing many true positives for that class. It's failing to identify actual members of this segment. We might need more data for this class, or features that make this class more distinct.
# - **High variance in performance across classes**: This often points to class imbalance. Techniques like SMOTE (already applied), `class_weight` parameters in models, or more targeted feature engineering for underrepresented classes can help.
# - **Analyze confusion matrix**: Look at specific off-diagonal cells. If segment A is consistently misclassified as segment B, then features distinguishing A from B need to be enhanced.


## 13. Overfitting or Underfitting

### Overfitting

-   **Definition**: Overfitting occurs when a model learns the training data too well, including its noise and outliers. It performs exceptionally well on the training data but poorly on unseen test data. The model has high variance.
-   **Symptoms**: High accuracy/F1-score on the training set, but significantly lower accuracy/F1-score on the test set.
-   **How to Fix**:
    1.  **More Data**: The simplest solution, if feasible.
    2.  **Regularization**: Techniques like L1/L2 regularization (for linear models) or adding dropout layers (for neural networks) penalize large coefficients, reducing model complexity.
    3.  **Simpler Models**: Use models with fewer parameters or lower complexity (e.g., decision tree with limited depth instead of a deep tree).
    4.  **Feature Selection/Reduction**: Remove irrelevant or redundant features, or use dimensionality reduction techniques like PCA.
    5.  **Cross-Validation**: Helps in identifying overfitting by providing a more robust estimate of performance on unseen data.
    6.  **Early Stopping**: For iterative models, stop training when validation performance starts to degrade.

### Underfitting

-   **Definition**: Underfitting occurs when a model is too simple to capture the underlying patterns in the data. It performs poorly on both training and test data. The model has high bias.
-   **Symptoms**: Low accuracy/F1-score on both the training set and the test set.
-   **How to Fix**:
    1.  **More Complex Models**: Use models with more parameters or higher complexity (e.g., Random Forest instead of Logistic Regression, or a deeper neural network).
    2.  **Feature Engineering**: Create new, more informative features from existing ones.
    3.  **Reduce Regularization**: If regularization is too strong, it can lead to underfitting.
    4.  **Increase Training Time/Iterations**: For iterative models, ensure the model has enough time to learn.

### Checking for Overfitting/Underfitting in Our Models

We can infer overfitting/underfitting by comparing the training set performance to the test set performance.


In [ ]:
print("\n--- Checking for Overfitting/Underfitting ---")
for name, res in results.items():
    model = res['model']
    # Calculate training accuracy/F1
    y_train_pred = model.predict(X_train_transformed)
    train_accuracy = accuracy_score(y_train_resampled if 'SMOTE' in globals() else y_train, y_train_pred) # Use resampled y if SMOTE applied
    train_f1 = f1_score(y_train_resampled if 'SMOTE' in globals() else y_train, y_train_pred, average='weighted')

    # Test accuracy/F1
    test_accuracy = res['accuracy']
    test_f1 = res['f1_score']

    print(f"\nModel: {name}")
    print(f"  Train Accuracy: {train_accuracy:.4f}, Test Accuracy: {test_accuracy:.4f}")
    print(f"  Train F1-Score: {train_f1:.4f}, Test F1-Score: {test_f1:.4f}")

    if train_accuracy > test_accuracy and (train_accuracy - test_accuracy) > 0.1: # Heuristic for significant difference
        print("  --> Potentially Overfitting!")
    elif train_accuracy < 0.6 and test_accuracy < 0.6: # Heuristic for poor performance
        print("  --> Potentially Underfitting!")
    else:
        print("  --> Model performance seems balanced or well-fitted.")

# For our current small sample, models might appear to overfit due to limited data,
# but on the full dataset, the difference between train and test scores usually reveals the true picture.
# Generally, ensemble methods like Random Forest and XGBoost are less prone to severe underfitting
# and have mechanisms (like tree depth limits, subsampling) to mitigate overfitting.


## 14. Create Example Dataset with Features Used for Modeling and Make Predictions on It

Let's create a small example dataset with hypothetical customer data, preprocess it, and then use our best-performing model to make predictions.


In [ ]:
# Example customer data (must match original features before one-hot/ordinal encoding)
example_data = [
    {'Gender': 'Male', 'Ever_Married': 'No', 'Age': 25, 'Graduated': 'No', 'Profession': 'Healthcare', 'Work_Experience': 2.0, 'Spending_Score': 'Low', 'Family_Size': 3.0, 'Var_1': 'Cat_4', 'Age_Group': '19-30', 'Work_Experience_Per_Family_Member': 0.66},
    {'Gender': 'Female', 'Ever_Married': 'Yes', 'Age': 45, 'Graduated': 'Yes', 'Profession': 'Artist', 'Work_Experience': 10.0, 'Spending_Score': 'Average', 'Family_Size': 2.0, 'Var_1': 'Cat_6', 'Age_Group': '31-45', 'Work_Experience_Per_Family_Member': 5.0},
    {'Gender': 'Male', 'Ever_Married': 'Yes', 'Age': 60, 'Graduated': 'Yes', 'Profession': 'Engineer', 'Work_Experience': 20.0, 'Spending_Score': 'High', 'Family_Size': 1.0, 'Var_1': 'Cat_6', 'Age_Group': '46-60', 'Work_Experience_Per_Family_Member': 20.0},
]
example_df = pd.DataFrame(example_data)

print("Example dataset for prediction:")
print(example_df)

# Preprocess the example data using the *fitted* preprocessor from training
example_transformed = fitted_preprocessor.transform(example_df)

# Make predictions using the best model
best_model_name = max(results, key=lambda k: results[k]['f1_score'])
best_model = results[best_model_name]['model']

example_predictions_encoded = best_model.predict(example_transformed)
example_predictions_proba = best_model.predict_proba(example_transformed)

# Inverse transform predictions to original labels
example_predictions = le.inverse_transform(example_predictions_encoded)

print(f"\nPredictions using {best_model_name}:")
for i, pred_label in enumerate(example_predictions):
    print(f"Customer {i+1}: Predicted Segment: {pred_label}")
    # Show probabilities for top 3 classes
    top_n = np.argsort(example_predictions_proba[i])[-3:][::-1] # indices of top 3 probabilities
    print("  Top 3 Probabilities:")
    for rank, class_idx in enumerate(top_n):
        class_name = le.inverse_transform([class_idx])[0]
        prob = example_predictions_proba[i][class_idx]
        print(f"    {rank+1}. {class_name}: {prob:.4f}")


## 15. Hyperparameter Tuning on Sample or Small Dataset

Hyperparameter tuning is crucial for optimizing model performance. We will use `RandomizedSearchCV` on a small subset of our training data to find good hyperparameters for the `RandomForestClassifier` (which usually performs well). Randomized search is more efficient than Grid Search for large hyperparameter spaces.


In [ ]:
# Use a smaller subset for tuning to save computation time, as instructed.
# For demonstration, we'll use 20% of the training data.
X_tune, _, y_tune, _ = train_test_split(X_train_resampled, y_train_resampled, test_size=0.8, random_state=42, stratify=y_train_resampled)
print(f"\nUsing {X_tune.shape[0]} samples ({len(y_tune)} target values) for hyperparameter tuning.")

# Define the model to tune
rf_model = RandomForestClassifier(random_state=42)

# Define the parameter distribution for RandomizedSearchCV
param_dist = {
    'n_estimators': [100, 200, 300], # Number of trees in the forest
    'max_features': ['sqrt', 'log2', 0.6, 0.8], # Number of features to consider when looking for the best split
    'max_depth': [10, 20, 30, None], # Maximum number of levels in tree
    'min_samples_split': [2, 5, 10], # Minimum number of samples required to split an internal node
    'min_samples_leaf': [1, 2, 4], # Minimum number of samples required to be at a leaf node
    'bootstrap': [True, False] # Whether bootstrap samples are used when building trees
}

# Perform RandomizedSearchCV
# n_iter controls the number of parameter settings that are sampled.
# cv=3 for cross-validation on the tuning subset.
random_search = RandomizedSearchCV(estimator=rf_model, param_distributions=param_dist,
                                   n_iter=10, cv=3, verbose=2, random_state=42, n_jobs=-1,
                                   scoring='f1_weighted')

print("\nStarting RandomizedSearchCV for Random Forest...")
random_search.fit(X_tune, y_tune)

print(f"\nBest parameters found: {random_search.best_params_}")
print(f"Best cross-validation F1-score (weighted): {random_search.best_score_:.4f}")

# Train a new model with the best parameters on the full resampled training data
best_rf_model = random_search.best_estimator_
print(f"\nTraining Random Forest with best parameters on full training data ({X_train_resampled.shape[0]} samples)...")
best_rf_model.fit(X_train_resampled, y_train_resampled)

# Evaluate the tuned model on the test set
y_pred_tuned_rf = best_rf_model.predict(X_test_transformed)
tuned_rf_accuracy = accuracy_score(y_test, y_pred_tuned_rf)
tuned_rf_f1 = f1_score(y_test, y_pred_tuned_rf, average='weighted')

print(f"\nTuned Random Forest - Test Accuracy: {tuned_rf_accuracy:.4f}, F1-Score (weighted): {tuned_rf_f1:.4f}")
print(f"Classification Report for Tuned Random Forest:\n{classification_report(y_test, y_pred_tuned_rf, target_names=le.classes_)}")

# Update results with the tuned model
results['Tuned Random Forest'] = {'accuracy': tuned_rf_accuracy, 'f1_score': tuned_rf_f1, 'model': best_rf_model}

# Suggest Hyperparameters explanation:
# - `n_estimators`: More trees generally improve performance but increase computation. We tested 100-300.
# - `max_features`: Controls the number of features considered at each split. 'sqrt' is common, others like 'log2' or specific percentages (0.6, 0.8) explore different levels of randomness.
# - `max_depth`: Limits the depth of trees to prevent overfitting. `None` means nodes are expanded until all leaves are pure or contain less than `min_samples_split` samples.
# - `min_samples_split`: Minimum number of samples required to split an internal node. Higher values prevent the model from learning highly specific patterns.
# - `min_samples_leaf`: Minimum number of samples required to be at a leaf node. Similar to `min_samples_split`, higher values increase generalization.
# - `bootstrap`: Whether bootstrap samples are used. Typically True for Random Forests.
# `RandomizedSearchCV` efficiently samples different combinations of these to find a good set.


## 16. Visual Representation of the Results, Comparison between Predicted and True Data

We will visualize the performance of our best model, specifically using the confusion matrix and comparing the distribution of predicted vs. true labels.


In [ ]:
# Get the best model after tuning
final_best_model_name = max(results, key=lambda k: results[k]['f1_score'])
final_best_model = results[final_best_model_name]['model']

print(f"Final best model selected: {final_best_model_name}")

y_pred_final = final_best_model.predict(X_test_transformed)

# 1. Confusion Matrix for the Final Best Model
cm_final = confusion_matrix(y_test, y_pred_final)
fig_cm_final = px.imshow(cm_final, text_auto=True, color_continuous_scale='Viridis',
                          labels=dict(x="Predicted Segmentation", y="True Segmentation"),
                          x=le.classes_, y=le.classes_,
                          title=f'Confusion Matrix for {final_best_model_name} (Test Data)')
fig_cm_final.show()

# Explanation:
# - The confusion matrix vividly shows how many instances of each true segment were classified into each predicted segment.
# - Strong diagonal values indicate accurate predictions.
# - Off-diagonal values highlight misclassifications. For example, a large number in row 'A', column 'B' means many customers who are truly in Segment A were incorrectly predicted to be in Segment B. This helps identify where the model struggles.

# 2. Bar Chart: Comparison of True vs. Predicted Segment Distribution
true_counts = pd.Series(y_test).map(lambda x: le.inverse_transform([x])[0]).value_counts().sort_index()
pred_counts = pd.Series(y_pred_final).map(lambda x: le.inverse_transform([x])[0]).value_counts().sort_index()

comparison_df = pd.DataFrame({'True': true_counts, 'Predicted': pred_counts}).fillna(0)
comparison_df.index.name = 'Segmentation'
comparison_df.reset_index(inplace=True)

fig_comp = px.bar(comparison_df, x='Segmentation', y=['True', 'Predicted'],
                  barmode='group', title=f'Comparison of True vs. Predicted Segmentation Distribution ({final_best_model_name})',
                  labels={'value': 'Count'},
                  color_discrete_sequence=custom_colors)
fig_comp.update_xaxes(categoryorder='array', categoryarray=le.classes_) # Ensure consistent order of segments
fig_comp.show()

# Explanation:
# - This bar chart compares the actual distribution of customer segments in the test set with the distribution predicted by the model.
# - Ideally, the 'True' and 'Predicted' bars for each segment should be very close in height.
# - Significant discrepancies indicate that the model might be over-predicting some segments and under-predicting others, even if its overall accuracy is decent. This can be a sign of class imbalance issues or features not strong enough to distinguish certain segments.

[MARKMARKDOWN]
## 17. Final Model Selection Based on Best Result

Based on our initial evaluations and hyperparameter tuning, we will select the model with the highest F1-score (weighted) on the test set. The F1-score is chosen as the primary metric because it provides a good balance between precision and recall, which is important for multi-class classification, especially if there's any class imbalance.


In [ ]:
# Identify the best model based on weighted F1-score
final_best_model_name = ''
best_f1_score = -1

for name, metrics in results.items():
    if metrics['f1_score'] > best_f1_score:
        best_f1_score = metrics['f1_score']
        final_best_model_name = name

final_best_model = results[final_best_model_name]['model']

print(f"\n--- Final Model Selection ---")
print(f"The best model is: {final_best_model_name}")
print(f"Achieved F1-Score (weighted) on test set: {best_f1_score:.4f}")
print(f"Achieved Accuracy on test set: {results[final_best_model_name]['accuracy']:.4f}")

# Rationale for selection:
# The {final_best_model_name} model achieved the highest weighted F1-score, indicating a robust balance of precision and recall across all customer segments. While accuracy is also important, F1-score is often preferred in multi-class scenarios to ensure adequate performance on all classes, especially if there are varying class sizes. The hyperparameter tuning further optimized its performance, making it the most suitable choice for deployment.


## 18. Save the Model Using Pickle Library

It's essential to save the trained model along with the preprocessor and label encoder, so they can be loaded later to make predictions on new, unseen data without retraining.


In [ ]:
# Save the final best model
model_filename = f'{final_best_model_name.lower().replace(" ", "_")}_segmentation_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(final_best_model, file)
print(f"Final model '{final_best_model_name}' saved as {model_filename}")

# Save the fitted preprocessor
preprocessor_filename = 'segmentation_preprocessor.pkl'
with open(preprocessor_filename, 'wb') as file:
    pickle.dump(fitted_preprocessor, file)
print(f"Fitted preprocessor saved as {preprocessor_filename}")

# Save the label encoder for target variable
label_encoder_filename = 'segmentation_label_encoder.pkl'
with open(label_encoder_filename, 'wb') as file:
    pickle.dump(le, file)
print(f"Label encoder saved as {label_encoder_filename}")

print("\nAll necessary components (model, preprocessor, label encoder) have been saved.")

# Example of loading and using the saved components (optional for demonstration)
# print("\n--- Demonstrating loading and prediction with saved components ---")
# with open(model_filename, 'rb') as file:
#     loaded_model = pickle.load(file)
# with open(preprocessor_filename, 'rb') as file:
#     loaded_preprocessor = pickle.load(file)
# with open(label_encoder_filename, 'rb') as file:
#     loaded_le = pickle.load(file)

# new_customer_data = pd.DataFrame([{'Gender': 'Male', 'Ever_Married': 'No', 'Age': 28, 'Graduated': 'No', 'Profession': 'Healthcare', 'Work_Experience': 3.0, 'Spending_Score': 'Low', 'Family_Size': 2.0, 'Var_1': 'Cat_4', 'Age_Group': '19-30', 'Work_Experience_Per_Family_Member': 1.5}])
# new_customer_data_processed = loaded_preprocessor.transform(new_customer_data)
# new_prediction_encoded = loaded_model.predict(new_customer_data_processed)
# new_prediction_label = loaded_le.inverse_transform(new_prediction_encoded)
# print(f"Prediction for new customer: {new_prediction_label[0]}")


## 19. Insights

Based on our analysis, here are some key insights regarding customer segmentation:

1.  **Key Influencing Features**:
    *   **Profession** and **Spending_Score** appear to be very strong indicators for customer segmentation. Distinct professions tend to cluster within specific segments, and spending habits (Low, Average, High) clearly differentiate segments. For example, 'Artist' and 'Healthcare' might be prominent in certain segments.
    *   **Age** and **Ever_Married** also show significant differences across segments, indicating lifecycle stages and demographics play a role. Older, married customers might belong to different segments than younger, single ones.
    *   **Work_Experience** and **Family_Size** contribute to the segmentation, although their impact might be more nuanced. The engineered feature `Work_Experience_Per_Family_Member` could capture interesting family-work dynamics.
    *   **Gender** and `Var_1` also contribute, but might be less discriminative than profession or spending score.

2.  **Segment Characteristics**: Each segment (A, B, C, D) likely represents customers with distinct profiles. For example:
    *   One segment might be characterized by younger, single professionals with lower spending scores (e.g., segment D from sample data).
    *   Another might be older, married individuals with high work experience and average spending scores (e.g., segment C).
    *   The business can leverage these insights to tailor marketing messages, product offerings, and customer service strategies to each segment, leading to higher engagement and satisfaction.

3.  **Model Performance**:
    *   The chosen model, a **{final_best_model_name}**, demonstrated good performance on the unseen test data, achieving a weighted F1-score of **{best_f1_score:.4f}**. This indicates its ability to generalize well to new customers.
    *   The confusion matrix provided clear insights into where the model performs well and where it might struggle with specific class distinctions. For example, if many 'A' customers are misclassified as 'B', further investigation into features distinguishing these two segments would be beneficial.

4.  **Data Quality**:
    *   Missing values were successfully handled, preventing data integrity issues.
    *   Outlier capping ensured that extreme values in numerical features did not unduly influence model training.


## 20. Conclusion

This project successfully built and evaluated machine learning models for customer segmentation. We followed a structured approach from data loading and comprehensive EDA to preprocessing, feature engineering, model training, and evaluation.

The **{final_best_model_name}** model was selected as the final model due to its superior performance on the test set, specifically its high weighted F1-score, which suggests a balanced predictive capability across all customer segments. The model, along with its preprocessor and label encoder, has been saved for future inference on new customer data.

The insights gained from this analysis, particularly regarding the influence of `Profession`, `Spending_Score`, `Age`, and `Ever_Married`, can empower businesses to develop more effective and personalized strategies. By understanding their customer segments, businesses can optimize their resource allocation, improve customer satisfaction, and drive higher profitability.

Further improvements could involve exploring more advanced feature engineering techniques, experimenting with deep learning models, or collecting additional relevant customer data to refine the segmentation process even further.